In [ ]:
# El mismo caso de P005 permite comparar decisiones de limpieza en Python y en SQL.

from pathlib import Path
import json
import sqlite3

import pandas as pd

ACTIVITY_DIR = next(candidate for candidate in [Path.cwd().resolve(), *Path.cwd().resolve().parents] if (candidate / "data").is_dir() and (candidate / "submission").is_dir())
DATA_FILE = ACTIVITY_DIR / "data" / "ventas.csv"
DATABASE_FILE = ACTIVITY_DIR / "temp" / "ventas.db"
SUBMISSION_DIR = ACTIVITY_DIR / "submission"

In [ ]:
# CSV sin limpiar ──► raw_sales ──► cleaned_sales ──► entrega analítica

raw_sales = pd.read_csv(DATA_FILE)
raw_sales.columns = [column.strip().lower().replace(" ", "_") for column in raw_sales.columns]
DATABASE_FILE.unlink(missing_ok=True)
database = sqlite3.connect(DATABASE_FILE)
raw_sales.to_sql("raw_sales", database, index=False)

In [ ]:
# Normalizar texto y códigos evita que una misma entidad aparezca fragmentada en el análisis.

database.execute("""
    CREATE TABLE cleaned_sales AS
    SELECT
        supplier_id,
        TRIM(supplier) AS supplier,
        'COL' AS country,
        CASE
            WHEN LOWER(TRIM(city)) IN ('bogotá', 'bogota') THEN 'Bogotá'
            WHEN LOWER(TRIM(city)) IN ('medellín', 'medellin') THEN 'Medellín'
            ELSE TRIM(city)
        END AS city,
        REPLACE(REPLACE(TRIM(purchase_date), '/', '-'), '.', '-') AS purchase_date,
        CAST(REPLACE(REPLACE(REPLACE(TRIM(amount), 'COP ', ''), '$', ''), ',', '') AS REAL) AS amount,
        CASE
            WHEN discount IS NULL OR TRIM(discount) = '' THEN NULL
            WHEN INSTR(discount, '%') > 0 THEN CAST(REPLACE(discount, '%', '') AS REAL) / 100
            ELSE CAST(discount AS REAL)
        END AS discount,
        units,
        CAST(REPLACE(REPLACE(TRIM(unit_price), '$', ''), ',', '') AS REAL) AS unit_price,
        TRIM(contact_email) AS contact_email
    FROM raw_sales
""")

In [ ]:
# El reporte convierte la limpieza en una evidencia auditable, no en una transformación oculta.

quality_report = pd.read_sql_query("""
    SELECT
        COUNT(*) AS row_count,
        COUNT(DISTINCT supplier) AS supplier_count,
        SUM(CASE WHEN discount IS NULL THEN 1 ELSE 0 END) AS missing_discount_count,
        SUM(CASE WHEN amount IS NULL THEN 1 ELSE 0 END) AS missing_amount_count
    FROM cleaned_sales
""", database)
ventas_limpias = pd.read_sql_query("SELECT * FROM cleaned_sales ORDER BY supplier_id", database)
ventas_limpias.to_csv(SUBMISSION_DIR / "ventas_limpias.csv", index=False)
quality_report.to_csv(SUBMISSION_DIR / "quality_report.csv", index=False)
quality_report

In [ ]:
# Los resultados se conservan para comparar el caso limpio con la solución de P005.

questions = [{"pregunta": "¿Qué calidad tiene la tabla de ventas después de normalizar sus campos en SQL?", "archivo_respuesta": "quality_report.csv"}]
(SUBMISSION_DIR / "questions.json").write_text(json.dumps(questions, ensure_ascii=False, indent=2), encoding="utf-8")
database.close()